In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:

# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_lineitem, load_orders, load_customer, load_nation, q10
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:

### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)



In [6]:

### cell 2 ###


def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)


In [7]:

### cell 3 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [8]:
# %%time
# ### cell 4 ###

# import datetime

# # Define date bounds
# date1 = datetime.datetime(1994, 11, 1)
# date2 = datetime.datetime(1995, 2, 1)

# # One‐chain of vectorized GPU operations: filter, project, join, compute revenue, aggregate & sort
# total = (
#     lineitem
#       .loc[lineitem.L_RETURNFLAG == 'R', ['L_ORDERKEY', 'L_EXTENDEDPRICE', 'L_DISCOUNT']]
#       .merge(
#           orders.loc[(orders.O_ORDERDATE >= date1) & (orders.O_ORDERDATE < date2), ['O_ORDERKEY', 'O_CUSTKEY']],
#           left_on='L_ORDERKEY',
#           right_on='O_ORDERKEY'
#       )
#       .merge(
#           customer[['C_CUSTKEY', 'C_NAME', 'C_ACCTBAL', 'C_PHONE', 'C_NATIONKEY', 'C_ADDRESS', 'C_COMMENT']],
#           left_on='O_CUSTKEY',
#           right_on='C_CUSTKEY'
#       )
#       .merge(
#           nation[['N_NATIONKEY', 'N_NAME']],
#           left_on='C_NATIONKEY',
#           right_on='N_NATIONKEY'
#       )
#       .assign(TMP=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
#       .groupby(
#           ['C_CUSTKEY', 'C_NAME', 'C_ACCTBAL', 'C_PHONE', 'N_NAME', 'C_ADDRESS', 'C_COMMENT'],
#           as_index=False,
#           sort=False
#       )['TMP']
#       .sum()
#       .sort_values('TMP', ascending=False)
# )
